GLSKF low rank tensor experiment

In [1]:
import numpy as np
from scipy.sparse import eye, csr_matrix
from scipy.linalg import inv, khatri_rao
import matplotlib.pyplot as plt
import os
import pandas as pd
import time

# 绘图字体设置
# 中文优先使用宋体，英文和数字可回退到 Times New Roman
plt.rcParams['font.sans-serif'] = ['SimSun', 'Times New Roman']
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.unicode_minus'] = False

def cov_matern(d, loghyper, x):
    ell = np.exp(loghyper[0])
    sf2 = np.exp(2 * loghyper[1])

    def f(t):
        if d == 1:
            return 1
        if d == 3:
            return 1 + t
        if d == 5:
            return 1 + t * (1 + t / 3)
        if d == 7:
            return 1 + t * (1 + t * (6 + t) / 15)
        raise ValueError('不支持的 Matern 阶数')

    def m(t):
        return f(t) * np.exp(-t)

    dist_sq = ((x[:, None] - x[None, :]) / ell) ** 2
    return sf2 * m(np.sqrt(d * dist_sq))

def bohman(loghyper, x):
    range_ = np.exp(loghyper[0])
    dis = np.abs(x[:, None] - x[None, :])
    r = np.minimum(dis / range_, 1)
    k = (1 - r) * np.cos(np.pi * r) + np.sin(np.pi * r) / np.pi
    k[k < 1e-16] = 0
    k[np.isnan(k)] = 0
    return k

def unfold(tensor, mode):
    return np.reshape(np.moveaxis(tensor, mode, 0), (tensor.shape[mode], -1), order='F')

def fold(mat, dim, mode):
    index = [mode]
    for i in range(dim.shape[0]):
        if i != mode:
            index.append(i)
    return np.moveaxis(np.reshape(mat, list(dim[index]), order='F'), 0, mode)

def kroneckerMVM(K3, K2, K1, vec, d1, d2, d3):
    temp1 = (K1 @ vec.reshape(d1, d2, d3, order='F').reshape(d1, -1)).reshape(d1, d2, d3)
    temp2 = (K2 @ temp1.transpose(1, 0, 2).reshape(d2, -1)).reshape(d2, d1, d3).transpose(1, 0, 2)
    temp3 = (K3 @ temp2.transpose(2, 0, 1).reshape(d3, -1)).reshape(d3, d1, d2).transpose(1, 2, 0)
    return temp3.ravel(order='F')

def Ap_operatorT(vec, maskT, KrU, KrU_T, Qu, rho, rank_num, mode_dim):
    x = vec.reshape(rank_num, mode_dim, order='F')
    temp = KrU @ x
    temp *= maskT
    ap1 = KrU_T @ temp
    ap2 = rho * (x @ Qu)
    return (ap1 + ap2).ravel(order='F')

def cg_factorT(Qu, rho, KrU, mask_matrixT, y_tilde, priorvalue, max_iter):
    rank_num, mode_dim = y_tilde.shape
    y_flat = y_tilde.ravel(order='F')
    x = priorvalue.copy()
    KrU_T = KrU.T
    ax = Ap_operatorT(x, mask_matrixT, KrU, KrU_T, Qu, rho, rank_num, mode_dim)
    r = y_flat - ax
    p = r.copy()
    rsold = np.dot(r, r)
    approxE = np.zeros(max_iter)

    for i in range(max_iter):
        ap = Ap_operatorT(p, mask_matrixT, KrU, KrU_T, Qu, rho, rank_num, mode_dim)
        alpha = rsold / np.dot(p, ap)
        x += alpha * p
        r -= alpha * ap
        rsnew = np.dot(r, r)
        approxE[i] = np.sqrt(rsnew)
        if approxE[i] < 1e-4:
            break
        p = r + (rsnew / rsold) * p
        rsold = rsnew
    return x, approxE

def Ap_operatorL(vec, pos_obs, Kd, Kt, Ks, gamma, d1, d2, d3):
    x = np.zeros(d1 * d2 * d3)
    x[pos_obs] = vec
    ap1 = kroneckerMVM(Kd, Kt, Ks, x, d1, d2, d3)
    return ap1[pos_obs] + gamma * vec

def cg_local(gamma, Kd, Kt, Ks, pos_obs, y_tilde, priorvalue, max_iter):
    d1, d2, d3 = y_tilde.shape
    y_obs = y_tilde.ravel(order='F')[pos_obs]
    x = priorvalue.copy()
    ax = Ap_operatorL(x, pos_obs, Kd, Kt, Ks, gamma, d1, d2, d3)
    r = y_obs - ax
    p = r.copy()
    rsold = np.dot(r, r)
    approxE = np.zeros(max_iter)

    for i in range(max_iter):
        ap = Ap_operatorL(p, pos_obs, Kd, Kt, Ks, gamma, d1, d2, d3)
        alpha = rsold / np.dot(p, ap)
        x += alpha * p
        r -= alpha * ap
        rsnew = np.dot(r, r)
        approxE[i] = np.sqrt(rsnew)
        if approxE[i] < 1e-4:
            break
        p = r + (rsnew / rsold) * p
        rsold = rsnew
    return x, approxE

def GLSKF(I, Omega, lengthscaleU, lengthscaleR, varianceU, varianceR, tapering_range, d_MaternU, d_MaternR, rank_num, rho, gamma, maxiter, K0, epsilon):
    N = np.array(I.shape)
    D = I.ndim
    maxP = float(np.max(I))

    Omega = Omega.astype(bool)
    pos_miss = np.where(~Omega)
    num_obser = np.sum(Omega)
    mask_matrix = [unfold(Omega, d) for d in range(D)]
    mask_matrixT = [mask_matrix[d].T for d in range(D)]
    pos_obs = np.where(mask_matrix[0].ravel(order='F') == 1)[0]
    idx_frame = np.sum(mask_matrix[2], axis=0) > 0

    train_values = I[Omega]
    train_mean = float(np.mean(train_values))
    Isubmean = I.astype(float) - train_mean
    T = Isubmean * Omega

    hyper_Ku = [None] * D
    hyper_Ku[0] = [np.log(lengthscaleU[0]), np.log(varianceU[0])]
    hyper_Ku[1] = [np.log(lengthscaleU[1]), np.log(varianceU[1])]

    hyper_Kr = [None] * D
    hyper_Kr[0] = [np.log(lengthscaleR[0]), np.log(varianceR[0]), np.log(tapering_range)]
    hyper_Kr[1] = [np.log(lengthscaleR[1]), np.log(varianceR[1]), np.log(tapering_range)]

    invKu = [None] * D
    Kr = [None] * D

    x = np.arange(1, N[0] + 1)
    Ku1 = cov_matern(d_MaternU, hyper_Ku[0], x)
    invKu[0] = inv(Ku1)
    taper1 = bohman([hyper_Kr[0][2]], x)
    Kr[0] = csr_matrix(cov_matern(d_MaternR, hyper_Kr[0][:2], x) * taper1)

    x = np.arange(1, N[1] + 1)
    Ku2 = cov_matern(d_MaternU, hyper_Ku[1], x)
    invKu[1] = inv(Ku2)
    taper2 = bohman([hyper_Kr[1][2]], x)
    Kr[1] = csr_matrix(cov_matern(d_MaternR, hyper_Kr[1][:2], x) * taper2)

    invKu[2] = csr_matrix(eye(N[2], format='csr'))
    Kr[2] = csr_matrix(eye(N[2], format='csr'))

    X = T.copy()
    X[pos_miss] = T.sum() / num_obser

    U = [0.1 * np.random.randn(N[d], rank_num) for d in range(D)]
    M_unfold1 = U[0] @ khatri_rao(U[2], U[1]).T
    M = fold(M_unfold1, N, 0)

    UTvector = [U[d].T.ravel(order='F') for d in range(D)]
    Rtensor = np.zeros(N, dtype=float)
    Rvector_temp = np.zeros(N[0] * N[1] * N[2], dtype=float)
    X[pos_miss] = M[pos_miss] + Rtensor[pos_miss]

    d_all = np.arange(0, D)
    train_norm = np.linalg.norm(T)
    last_tensor = T.copy()
    mse_hist = np.zeros(maxiter)
    rmse_hist = np.zeros(maxiter)
    rse_hist = np.zeros(maxiter)
    mae_hist = np.zeros(maxiter)
    elapsed_hist = np.zeros(maxiter)
    rel_hist = np.zeros(maxiter)
    iter_num = 0
    start_time = time.perf_counter()

    while True:
        Gtensor = X - Rtensor
        Gtensor_mask = Gtensor * Omega

        for d in range(D):
            dsub = np.delete(d_all, d)
            KrU = khatri_rao(U[dsub[1]], U[dsub[0]])
            HG = KrU.T @ unfold(Gtensor_mask, d).T
            UTvector[d], _ = cg_factorT(invKu[d], rho, KrU, mask_matrixT[d], HG, UTvector[d], 100)
            U[d] = UTvector[d].reshape(rank_num, N[d], order='F').T

        M_unfold1 = U[0] @ khatri_rao(U[2], U[1]).T
        M = fold(M_unfold1, N, 0)
        X[pos_miss] = M[pos_miss] + Rtensor[pos_miss]

        if iter_num >= K0:
            Ltensor = X - M
            Ltensor_mask = Ltensor * Omega
            Rvector_temp[pos_obs], _ = cg_local(gamma, Kr[2], Kr[1], Kr[0], pos_obs, Ltensor_mask, Rvector_temp[pos_obs], 100)
            Rvector = kroneckerMVM(Kr[2], Kr[1], Kr[0], Rvector_temp, N[0], N[1], N[2])
            Rtensor = Rvector.reshape(N, order='F')
            Rtensor_unfold3 = unfold(Rtensor, 2)
            if np.sum(idx_frame) > 1:
                Rtensor_unfold3_obs = Rtensor_unfold3[:, idx_frame]
                Kr[2] = np.cov(Rtensor_unfold3_obs)
        else:
            Rtensor = np.zeros_like(Rtensor)

        X[pos_miss] = M[pos_miss] + Rtensor[pos_miss]
        Xori = X + train_mean
        Xrecovery = np.maximum(0, Xori)
        Xrecovery = np.minimum(maxP, Xrecovery)

        diff = I.astype(float) - Xrecovery
        mse_val = float(np.mean(diff ** 2))
        rmse_val = float(np.sqrt(mse_val))
        rse_val = float(np.linalg.norm(diff.ravel()) / max(np.linalg.norm(I.ravel()), np.finfo(float).eps))
        mae_val = float(np.mean(np.abs(diff)))
        tol = float(np.linalg.norm(X - last_tensor) / train_norm)
        last_tensor = X.copy()

        mse_hist[iter_num] = mse_val
        rmse_hist[iter_num] = rmse_val
        rse_hist[iter_num] = rse_val
        mae_hist[iter_num] = mae_val
        elapsed_hist[iter_num] = time.perf_counter() - start_time
        rel_hist[iter_num] = tol
        iter_num += 1
        print(f'Epoch = {iter_num}, MSE = {mse_val:.10f}, RMSE = {rmse_val:.10f}')

        if (tol < epsilon) or (iter_num >= maxiter):
            break

    history = pd.DataFrame({
        'iteration': np.arange(1, iter_num + 1),
        'elapsed_time_seconds': elapsed_hist[:iter_num],
        'MSE': mse_hist[:iter_num],
        'RMSE': rmse_hist[:iter_num],
        'RSE': rse_hist[:iter_num],
        'MAE': mae_hist[:iter_num],
        'relative_change': rel_hist[:iter_num],
    })
    return Xrecovery, history


In [2]:

from pathlib import Path
base_dir = Path.cwd()
data_dir = base_dir / 'data'
result_root = base_dir / 'results' / 'GLSKF'
result_root.mkdir(parents=True, exist_ok=True)
seed_list = [920, 921, 922]
miss_list = [0.8, 0.9, 0.95]
lengthscaleU = np.ones(2) * 30
varianceU = np.ones(2)
lengthscaleR = np.ones(2) * 5
varianceR = np.ones(2)
tapering_range = 10
d_MaternU = 3
d_MaternR = 3
rank_num = 10
maxiter = 20
K0 = 10
epsilon = 1e-4
rho_list = [1, 5, 10]
gamma_list = [1, 5, 10]
all_summary = []

def enrich_history(history, data_name, seed, missing_rate, rho, gamma):
    history = history.copy()
    history.insert(0, 'dataset', data_name)
    history.insert(1, 'method', 'GLSKF')
    history.insert(2, 'variant', 'default')
    history.insert(3, 'seed', seed)
    history.insert(4, 'missing_rate', missing_rate)
    history['parameter_settings'] = f'rho={rho}, gamma={gamma}, rank_num={rank_num}, maxiter={maxiter}, K0={K0}'
    history['convergence_status'] = 'ok'
    return history

for seed in seed_list:
    for missing_rate in miss_list:
        miss_tag = int(round(missing_rate * 100))
        data_name = f'S{seed}_miss{miss_tag}'
        data = np.load(data_dir / f'{data_name}.npz', allow_pickle=True)
        Xtrue = data['X'].astype(float)
        Omega = data['Omega'].astype(bool)
        Y = data['Y'].astype(float)
        result_dir = result_root / data_name
        result_dir.mkdir(parents=True, exist_ok=True)
        print('\n' + '=' * 60)
        print('Method: GLSKF')
        print(f'Data: {data_name}')
        print(f'Size: {Xtrue.shape}')
        print(f'Missing rate: {missing_rate:.2f}')
        print(f'Observed entries: {int(Omega.sum())} / {Omega.size}')
        print('=' * 60)
        best_mse = np.inf
        best = None
        summary_rows = []
        case_id = 0
        for rho in rho_list:
            for gamma in gamma_list:
                case_id += 1
                print(f'[{case_id}] rho={rho}, gamma={gamma}')
                start = time.perf_counter()
                Xhat, history = GLSKF(Xtrue, Omega, lengthscaleU, lengthscaleR, varianceU, varianceR, tapering_range, d_MaternU, d_MaternR, rank_num, rho, gamma, maxiter, K0, epsilon)
                elapsed = time.perf_counter() - start
                Xhat = np.clip(Xhat, 0, 1)
                diff = Xtrue - Xhat
                mse = float(np.mean(diff ** 2))
                rmse = float(np.sqrt(mse))
                rse = float(np.linalg.norm(diff.ravel()) / max(np.linalg.norm(Xtrue.ravel()), np.finfo(float).eps))
                mae = float(np.mean(np.abs(diff)))
                rel = float(history['relative_change'].iloc[-1]) if len(history) else np.nan
                history = enrich_history(history, data_name, seed, missing_rate, rho, gamma)
                history_file = result_dir / f'case_{case_id:03d}_history.csv'
                history.to_csv(history_file, index=False, encoding='utf-8-sig')
                row = {'Case': case_id, 'Data': data_name, 'Method': 'GLSKF', 'Variant': 'default', 'MissingRate': missing_rate, 'Seed': seed, 'rho': rho, 'gamma': gamma, 'MSE': mse, 'RMSE': rmse, 'RSE': rse, 'MAE': mae, 'RelativeChange': rel, 'Time': elapsed, 'HistoryFile': str(history_file)}
                summary_rows.append(row)
                print(f'MSE={mse:.10f}, RMSE={rmse:.10f}, RSE={rse:.10f}, Time={elapsed:.2f}s')
                if mse < best_mse:
                    best_mse = mse
                    best = {'rho': rho, 'gamma': gamma, 'Xhat': Xhat, 'history': history, **row}
                    print(f'*** New best MSE = {best_mse:.10f} ***')
        summary = pd.DataFrame(summary_rows)
        summary.to_csv(result_dir / 'summary.csv', index=False, encoding='utf-8-sig')
        summary.to_excel(result_dir / '\u5b9e\u9a8c\u603b\u7ed3.xlsx', index=False)
        best['history'].to_csv(result_dir / '\u6700\u4f73\u8fed\u4ee3.csv', index=False, encoding='utf-8-sig')
        best['history'].to_csv(result_dir / 'best_iteration_history.csv', index=False, encoding='utf-8-sig')
        np.savez_compressed(result_dir / 'result.npz', data_name=data_name, X=Xtrue, Omega=Omega, Y=Y, recovered=best['Xhat'], rho=best['rho'], gamma=best['gamma'], missing_rate=missing_rate, seed=seed)
        with open(result_dir / 'metrics.txt', 'w', encoding='utf-8') as f:
            f.write(f'data={data_name}\nmethod=GLSKF\nvariant=default\nseed={seed}\nmissing_rate={missing_rate:.2f}\n')
            f.write(f'rho={best["rho"]}\ngamma={best["gamma"]}\nMSE={best["MSE"]:.10f}\nRMSE={best["RMSE"]:.10f}\nRSE={best["RSE"]:.10f}\nMAE={best["MAE"]:.10f}\nrelative_change={best["RelativeChange"]:.10f}\ntime={best["Time"]:.6f}\nstatus=ok\n')
        all_summary.append({k: best[k] for k in ['Data','Method','Variant','MissingRate','Seed','rho','gamma','MSE','RMSE','RSE','MAE','RelativeChange','Time']})
all_table = pd.DataFrame(all_summary)
all_table.to_csv(result_root / 'all_summary.csv', index=False, encoding='utf-8-sig')
all_table.to_excel(result_root / '\u5168\u90e8\u5b9e\u9a8c\u603b\u7ed3.xlsx', index=False)
print('All GLSKF low rank tensor experiments finished.')



Method: GLSKF
Data: S920_miss80
Size: (120, 120, 30)
Missing rate: 0.80
Observed entries: 86210 / 432000
[1] rho=1, gamma=1
Epoch = 1, MSE = 0.0075294280, RMSE = 0.0867722768
Epoch = 2, MSE = 0.0075294300, RMSE = 0.0867722881
Epoch = 3, MSE = 0.0075294286, RMSE = 0.0867722803
MSE=0.0075294286, RMSE=0.0867722803, RSE=0.1707669143, Time=1.10s
*** New best MSE = 0.0075294286 ***
[2] rho=1, gamma=5
Epoch = 1, MSE = 0.0075294394, RMSE = 0.0867723425
Epoch = 2, MSE = 0.0075294308, RMSE = 0.0867722930
Epoch = 3, MSE = 0.0075294285, RMSE = 0.0867722795
MSE=0.0075294285, RMSE=0.0867722795, RSE=0.1707669126, Time=1.05s
*** New best MSE = 0.0075294285 ***
[3] rho=1, gamma=10
Epoch = 1, MSE = 0.0075294310, RMSE = 0.0867722939
Epoch = 2, MSE = 0.0075294294, RMSE = 0.0867722849
Epoch = 3, MSE = 0.0075294269, RMSE = 0.0867722706
MSE=0.0075294269, RMSE=0.0867722706, RSE=0.1707668951, Time=0.99s
*** New best MSE = 0.0075294269 ***
[4] rho=5, gamma=1
Epoch = 1, MSE = 0.0075294258, RMSE = 0.0867722641
M